# The factor view, and the residual between the two views

*A learning exercise performed in role: a simulated mandate with no client and no institution. Nothing in this notebook is investment advice, a recommendation, or a client communication.*

**Entry point** `python3 -m portfolio_workbench.attribute.factor`

**Modules covered** `attribute/factor.py`

_Generated from the code by `python3 -m reporting.notebooks`: the module headers below are read out of the modules themselves, and the run is the entry point's own output._

## 1. What this module does, and the source of every method in it

The factor view explains the same active return the holding-based view decomposes, as the sum of the passive exposures the book carries, the active loadings it took, and its alpha. The entry point prints both views side by side with the residual between them, and refuses a skill claim for the worked fund example on the stated power argument.

**Sources.** Every public function of the modules this notebook covers, and what it traces to. The map is checked over the code by the acceptance fixture, so a method added without a source fails a command rather than going unnoticed.

- `attribute/factor.py::cross_view` traces to the cross-view identity of this effort: both views are taken complete, so the residual is zero by construction and fires only when one view reads a different month set
- `attribute/factor.py::decomposition` traces to the factor view of the same active return, read on the basis the coefficients were fitted in - a construction of this effort, not a published decomposition, and the basis convention is what makes it checkable against the holding-based total
- `attribute/factor.py::design` traces to the factor view of the same active return, read on the basis the coefficients were fitted in - a construction of this effort, not a published decomposition, and the basis convention is what makes it checkable against the holding-based total
- `attribute/factor.py::design_covariance` traces to the factor view of the same active return, read on the basis the coefficients were fitted in - a construction of this effort, not a published decomposition, and the basis convention is what makes it checkable against the holding-based total
- `attribute/factor.py::fund_example` traces to the worked decomposition of a fund's track record, whose skill claim is refused on the evaluation design's own power argument rather than on the fit
- `attribute/factor.py::main` traces to the factor view of the same active return, read on the basis the coefficients were fitted in - a construction of this effort, not a published decomposition, and the basis convention is what makes it checkable against the holding-based total
- `attribute/factor.py::report` traces to the factor view of the same active return, read on the basis the coefficients were fitted in - a construction of this effort, not a published decomposition, and the basis convention is what makes it checkable against the holding-based total
- `attribute/factor.py::risk_attribution` traces to the factor view of the same active return, read on the basis the coefficients were fitted in - a construction of this effort, not a published decomposition, and the basis convention is what makes it checkable against the holding-based total

## 2. Why it works this way, including what was rejected

_The module headers, verbatim: each records why the module is shaped the way it is, what was rejected, and the measurement that settled it. They are quoted here rather than restated, so the notebook cannot drift from the code._

**`attribute/factor.py`**

Factor attribution: the explanation layer beside the holding-based one, and the gap named.

**Two decompositions of one number, and neither is ever added to the other.** The holding-based
Brinson decomposition of `attribute/brinson.py` is what the committee reads: it reconciles to the
benchmark by construction and speaks the mandate's language. This is the explanation layer - *why* the
allocation effects came out as they did - and it decomposes the **same** active return into factor
exposures times factor returns, plus alpha, plus residual. The only legitimate sum in the system is
that total, so the two views are printed side by side and never summed; the difference between them is
a single named cross-view residual, and the word "reconciles" is used only where that residual is
inside a stated tolerance.

**The residual is a definition check, not a modelling error.** Both views decompose the same series, so
the cross-view residual is zero by construction and the case proves the definition rather than the
arithmetic - which is worth having precisely because it fires when one view is silently reading a
different month set, a different weight path or a different return frame. It is reported as a number
with its scale beside it, for the same reason every other residual in this package is.

**The coefficients are read on the basis they were fitted in, and the basis travels with the month.**
The block is fitted net of its predecessors inside each window, so the loadings on its columns belong
to the orthogonalised series rather than the declared ones. Attributing a traded month therefore needs
that month on the same basis, and the window's own map is the only object that carries a basis across
the boundary between the window and the month it is applied to. The map is a linear functional once the
window is fixed, exactly as a loading is, so applying it to a month outside the window is the same kind
of extrapolation the attribution already makes with every coefficient.

**The four construction sleeves are the block, so their return is attributed to the block in full.**
Their loadings on the published spine are a summary of the same return expressed in another basis - a
sensitivity, and a useful one - not a second decomposition of it; adding both would count their return
twice. Their identity loadings are moved onto the orthogonal basis here so that every sleeve in the
report is read against the same series, and their residual is exactly zero because their model is.

**Risk attribution is factor-based only**, because Brinson is a return decomposition with no risk
dimension: the factor and idiosyncratic parts of tracking error are read off the same
`factors.exposures.factor_split` the factor layer reports, so the split the budget quotes and the split
this module quotes cannot come apart.

**The fund decomposition is a worked example and its skill claim is refused.** One book is read as
though it were an external manager's track record and its residual tested against the family-wise bar.
The finding is stated before the analysis: on 131 months with a few factors, a residual t-statistic
cannot clear that bar for a fund, so the method is demonstrated and the claim is not made.

## 3. The data contract it consumes, and the as-of rule

Coefficients are read on the basis they were fitted in: the block net of its predecessors inside the window, with the window's own map carrying that basis to a month outside it. Both views are taken **complete**, so the cross-view residual is zero by construction and fires only when one view reads a different month set. The unexplained part is reported as its own measured quantity rather than absorbed into that residual.

## 4. The worked example on small numbers, with the identity checked

The cross-view residual is an identity, so the worked example checks it directly: two views of one return that agree produce a zero residual, and a misaligned view makes it fire.

The cell below runs on numbers small enough to check by hand and asserts the identity, so a reader can see the arithmetic rather than take the module's word for it.

In [1]:
import pandas as pd

from portfolio_workbench.attribute import factor

# Two views that decompose the same active series agree exactly; one that reads a different month
# set does not, which is the only thing this residual is allowed to detect.
months = pd.PeriodIndex(["2020-01", "2020-02", "2020-03"], freq="M")
holding = pd.Series([0.002, -0.001, 0.003], index=months)

# Two views that decompose the same active series agree exactly.
agreement = factor.cross_view(holding, holding)
assert agreement["worst"] == 0.0 and agreement["reconciles"] is True

# A view that reads a different month set makes the residual fire, which is the only thing it detects.
misaligned = pd.Series([0.002, -0.001, 0.000], index=months)
assert factor.cross_view(holding, misaligned)["reconciles"] is False
print("aligned views gap", agreement["worst"], "; a misaligned view fires at", factor.cross_view(holding, misaligned)["worst"])

aligned views gap 0.0 ; a misaligned view fires at 0.003


## 5. The real run: inputs, parameters, provenance block

The provenance block is printed first, then the parameters this module decides under, then the entry point's own report. The report is the module's output rather than a transcription of it, so a number quoted from a notebook is the number the module prints.

In [2]:
from portfolio_workbench.data import loader, universe

document = loader.load_panel()
months = document.months
print(f"snapshot {document.snapshot_id}, taken as of {document.as_of}")
print(f"panel {len(months)} months {months.min()}..{months.max()} across {len(universe.TICKERS)} sleeves")
print("manifest fields: " + ", ".join(sorted(document.manifest)))

from portfolio_workbench.attribute import factor

print("the two views are never added to each other; they explain one active return and are compared")

snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
panel 191 months 2010-09..2026-07 across 11 sleeves
manifest fields: created, excluded, files, instruments, snapshot_id, window
the two views are never added to each other; they explain one active return and are compared


In [3]:
import subprocess
import sys

finished = subprocess.run(
    [sys.executable, "-m", "portfolio_workbench.attribute.factor"], capture_output=True, text=True, cwd="."
)
print(finished.stdout)
assert finished.returncode == 0, finished.stderr

[attrib] snapshot 2026-09-13: 131 refits 2015-09..2026-07 on 10 factors
[attrib] two views of one active return, never added to each other: the holding-based Brinson total from Brinson & Fachler (1985), JPM 11(3), 73-76, and this factor decomposition as the explanation
[attrib] cell                               brinson-sum    factors     alpha  unexplained  cross-view
[attrib] equal_weight                         -10.7250%   -5.6138%  -4.2468%     -0.8644%    1.13e-17
[attrib] policy                                +0.0000%   +0.0000%  +0.0000%     +0.0000%    0.00e+00
[attrib] mean_variance_shrunk                 +30.1058%  +34.4238% +11.4792%    -15.7972%    5.55e-17
[attrib] minimum_variance                     -51.7770%  -48.3063%  +0.8593%     -4.3300%    1.39e-17
[attrib] maximum_diversification              -33.9204%  -38.3303%  +3.7899%     +0.6200%    1.44e-15
[attrib] erc_unbounded                        -58.0335%  -50.9762%  -1.6837%     -5.3735%    2.78e-17
[attrib] erc_bou

## 6. Results, and how to read them, including the resolution limit and what a reader must not conclude

The residual between the views is zero by construction, so its value is not the reading: what a reader reads is the unexplained part, which is the model's out-of-sample residual and is measured rather than hidden. Alpha on a cell is a residual after the factors, and its resolution limit is the detectable-alpha bar the worked example prints. A reader must not read the alpha as skill: the fund example's own alpha is refused precisely because the panel's power cannot separate it from zero.

## 7. What this module does not establish

Nothing here establishes that the factor set explains the active return in a causal sense, or that alpha is stable out of sample. The two views are two accounts of one series, not two independent confirmations, and the unexplained part is a limit of the model rather than a market fact.